In [ ]:
# =============================================================================
# CELL 1 - COLAB ENVIRONMENT SETUP
# =============================================================================
# Installs, GPU check, Google Drive mount and Hugging Face login.
# facebook/sam3 is a GATED model: you must accept the license at
# https://huggingface.co/facebook/sam3 with the same account as your token.
# =============================================================================

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!pip install -q -U transformers accelerate huggingface_hub supervision psutil

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # uses HF_TOKEN from Colab secrets, or paste your token (needs access to facebook/sam3)

print("Environment ready.")


In [ ]:
# =============================================================================
# CELL 2 - IMPORTS
# =============================================================================

import os
import gc
import csv
import time
import json
import hashlib

import numpy as np
import pandas as pd
import torch
import psutil

from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

import supervision as sv
from supervision.metrics import MeanAveragePrecision

from transformers import Sam3Model, Sam3Processor

# UAV orthomosaics are very large; disable PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

print("Imports OK. supervision version:", sv.__version__)


In [ ]:
# =============================================================================
# CELL 3 - CONFIGURATION
# =============================================================================
# Every parameter of the study lives here so that a thesis experiment is fully
# described by this single cell.
#
# E01_2c is E02_2's structure/evaluation protocol (bug-fixed strip composition,
# offline NMS, all_gt/held_out evaluation) applied to THIS notebook's own
# variant: single-exemplar tiling (N_EXEMPLARS=1) with this notebook's own
# tile size (TILE_SIZE=1536, OVERLAP=384) plus an opt-in downsampled
# whole-image "global context" pass, merged into the tiled detections before
# NMS (ADD_GLOBAL_CONTEXT_PASS). It is the single-exemplar sibling of E02_2c
# (which uses N_EXEMPLARS=3 and TILE_SIZE=3500, otherwise identical structure)
# and a smaller-tile relative of E01_2b (TILE_SIZE=2000, OVERLAP=800).
# Presence-score gating has been removed entirely (it was dead/commented code
# in the original notebook and was never actually applied).
# Named "E01_2c" (not "E01_2") so it writes to its own results folder instead
# of overwriting any earlier E01_2 or E01_2b run.
# =============================================================================

# ------------------------- experiment identity -------------------------------
EXPERIMENT_NAME = "E01_2c"    # distinct from any earlier "E01_2"/"E01_2b" run
N_EXEMPLARS     = 1           # single visual prompt (kept as-is for E01_2c)
USE_TILING      = True        # True  -> overlapping tiles
                              # False -> whole image treated as one single tile
PROMPT_TYPE = "multiple" if N_EXEMPLARS > 1 else "single"

# ------------------------------- dataset -------------------------------------
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0
VALID_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------- outputs -------------------------------------
RESULTS_ROOT = f"/content/drive/MyDrive/master_thesis/results_new1/{EXPERIMENT_NAME}"

# ------------------------------- tiling --------------------------------------
# Kept at this notebook's own values (smaller than the E01_2b/E02_2b sibling's
# 2000/800, but still above the E02_2 reference's 1000/150): TILE_SIZE
# comfortably exceeds the measured max plant width so a plant is never a huge
# fraction of its tile; OVERLAP is set to this notebook's own value (25% of
# TILE_SIZE). Note: OVERLAP should exceed the widest plant in the dataset so
# no plant is split across tiles -- worth checking against the measured plant
# width stats if you haven't already.
TILE_SIZE = 1536
OVERLAP   = 384
CACHE_TILES_IN_MEMORY = True  # True  -> crop all tiles once per image and keep the
                              #          PIL images in RAM (CPU) for all anchor runs
                              # False -> crop tiles on demand from the (already open)
                              #          full image; slower but uses less RAM

# --------------------------- SAM3 inference ----------------------------------
# SAM3 is executed EXACTLY ONCE per (image_ID, anchor_idx, tile) at this
# minimum score. Every higher "operating" confidence threshold is applied
# offline afterwards.
SAM3_INFERENCE_THRESHOLD = 0.30
MASK_THRESHOLD = 0.40         # SAM3 mask binarisation (only used transiently)
BATCH_SIZE = 4                # composed tiles per forward pass.
                              # TILE_SIZE=1536 tiles are smaller than the
                              # E01_2b/E02_2b sibling's 2000px tiles, so there's
                              # likely GPU headroom to raise this further.
USE_FP16 = True               # half precision inference on the GPU
KEEP_MASKS = False            # masks are NEVER stored (RAM / GPU / disk / CSV / NPZ).
                              # They are only used transiently to compute the
                              # mask-fill ratio of the plausibility filter.

# ------------------------- exemplar strip layout ------------------------------
STRIP_MARGIN = 6              # px between exemplar crops inside the strip
FEATHER_WIDTH = 8             # px of soft alpha blending around each exemplar crop
BACKGROUND_BLUR_RADIUS = 1.5  # light Gaussian blur applied to the sampled background
MAX_STRIP_HEIGHT_FRACTION = None
# ^ Optional extra clamp on the strip HEIGHT, expressed as a fraction of the tile
#   height. Disabled (None) by default, same as E02_2.

# ------------------- plausibility filter (aligned to E02_2) -------------------
MIN_FILL_RATIO   = 0.15       # >=15 % of the box area must be covered by the mask
MAX_AREA_FRACTION = 0.80      # a detection may not cover >80 % of a tile (was 0.60
                              # in this notebook's old code -> aligned to E02_2)
EDGE_MARGIN      = 5          # boxes with width or height <=5 px are discarded

# ---------------- target-region membership rule (bug-fixed, as in E02_2) ------
# A prediction made on the composed image (exemplar strip on top + tile below) is
# kept only if at least this fraction of its AREA lies inside the real tile region.
# Majority rule (0.50): the box belongs to whichever region holds most of its area.
# (Replaces this notebook's old "y1 >= dy - 5" top-edge-only rule.)
TILE_REGION_MIN_FRACTION = 0.50

# ------------------- optional: downsampled global-context pass ----------------
# An EXTRA single pass over the whole image, downscaled by GLOBAL_DOWNSCALE, run
# through the SAME exemplar-strip + SAM3 pipeline as a tile (no cropping). Its
# detections are rescaled back to full resolution and pooled together with the
# tiled detections BEFORE NMS -- so the frozen NMS/confidence operating point and
# every evaluation cell downstream need no special-casing at all.
ADD_GLOBAL_CONTEXT_PASS = True    # kept ON, same default as this notebook's own code
GLOBAL_DOWNSCALE = 2              # shrink factor for the global pass image


# ------------------------------ evaluation ------------------------------------
EVAL_IOU_THRESHOLD = 0.50     # IoU needed for a prediction to count as a TP
PROMPT_IGNORE_IOU  = 0.50     # held_out mode: an unmatched prediction whose best IoU
                              # with a PROMPT GT box is >= this value is IGNORED
                              # (neither TP nor FP). Same value as EVAL_IOU_THRESHOLD
                              # so a single IoU threshold governs the whole protocol.

# ------------------------ operating point (frozen) -----------------------------
# No offline confidence x NMS sweep is performed. Both values are fixed from the
# start and used directly wherever an operating point is needed -- same frozen
# values as E02_2 (this notebook's old code used THRESHOLD=0.3 directly as its
# only cutoff, and NMS_IOU_THRESHOLD=0.5; both are now aligned to E02_2).
BEST_CONFIDENCE = 0.40
BEST_NMS_IOU = 0.40

EVALUATION_MODES = ["all_gt", "held_out"]
# all_gt   : every GT box of the image is evaluated (classical evaluation).
# held_out : the GT instances that were used as visual prompts are IGNORED, and so
#            are the predictions that fall on them. Answers "how well does SAM3 find
#            the REMAINING Rumex plants after being shown a few examples?".

# ------------------------------ safety net -------------------------------------
# A global pass plus tiling still uses more host RAM than the E02_2 reference's
# 1000px tiles alone. This mirrors this notebook's original memory guard: the
# main loop (CELL 19) stops cleanly (not a hard crash) if system RAM exceeds
# this percentage.
MEM_STOP_THRESHOLD_PCT = 70

print(f"Configuration loaded for experiment '{EXPERIMENT_NAME}'")
print(f"  prompts        : {N_EXEMPLARS} ({PROMPT_TYPE})")
print(f"  tiling         : {USE_TILING} (tile={TILE_SIZE}, overlap={OVERLAP})")
print(f"  SAM3 threshold : {SAM3_INFERENCE_THRESHOLD} (single inference pass per tile)")
print(f"  batch / fp16   : {BATCH_SIZE} / {USE_FP16}")
print(f"  masks stored   : {KEEP_MASKS}")
print(f"  global pass    : {ADD_GLOBAL_CONTEXT_PASS} (downscale={GLOBAL_DOWNSCALE})")
print(f"  operating pt   : confidence={BEST_CONFIDENCE}, NMS IoU={BEST_NMS_IOU} (frozen, no sweep)")


In [ ]:
# =============================================================================
# CELL 4 - OUTPUT FOLDERS
# =============================================================================
# results_new1/<EXPERIMENT_NAME>/
#   raw_detections/      pre-NMS detections (NPZ, one file per image x anchor)
#   metrics/             run / image / experiment / dataset level CSVs
#   confusion_matrices/  CSV + PNG for all_gt and held_out
# =============================================================================

RAW_DETECTIONS_DIR    = os.path.join(RESULTS_ROOT, "raw_detections")
METRICS_DIR           = os.path.join(RESULTS_ROOT, "metrics")
CONFUSION_MATRIX_DIR  = os.path.join(RESULTS_ROOT, "confusion_matrices")

for d in [RESULTS_ROOT, RAW_DETECTIONS_DIR, METRICS_DIR, CONFUSION_MATRIX_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest of finished inference runs -> used for crash-safe resuming.
RUN_MANIFEST_CSV = os.path.join(RAW_DETECTIONS_DIR, "runs_manifest.csv")
MANIFEST_COLUMNS = [
    "experiment_name", "image_ID", "anchor_idx", "Prompt_ID", "Prompt_Type",
    "n_gt", "n_prompt_gt", "n_detections_pre_nms", "n_tiles", "used_global_pass",
    "image_width", "image_height", "npz_file", "inference_seconds",
]

print("Output folders ready under:", RESULTS_ROOT)


In [ ]:
# =============================================================================
# CELL 5 - SAM3 MODEL + PROCESSOR
# =============================================================================
# With USE_FP16 the weights are loaded in half precision, which roughly halves
# GPU memory and speeds up inference on a T4. Inference additionally runs inside
# torch.inference_mode() + torch.autocast (see CELL 15).
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if (USE_FP16 and device == "cuda") else torch.float32

sam3_model = Sam3Model.from_pretrained(
    "facebook/sam3",
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("SAM3 loaded.")
print("  model device:", next(sam3_model.parameters()).device)
print("  model dtype :", next(sam3_model.parameters()).dtype)

In [ ]:
# =============================================================================
# CELL 6 - STABLE REPRODUCIBILITY HELPERS
# =============================================================================
# WHY THIS EXISTS
# The previous version used Python's built-in hash() to seed the exemplar
# sampling. hash() of a str/tuple is randomised per interpreter process
# (PYTHONHASHSEED), so the SAME image + anchor could select DIFFERENT extra
# exemplars after a Colab restart or on another machine -> results were not
# reproducible.
#
# FIX: derive the seed from a SHA-256 digest of a plain text key. SHA-256 is
# a fixed mathematical function, therefore the same key always yields the same
# seed on every machine, every Python version and every session.
# =============================================================================

def stable_seed(*parts) -> int:
    """
    Deterministic 32-bit seed from any set of values.

    Input : any number of values (strings / ints) that identify the run,
            e.g. stable_seed(EXPERIMENT_NAME, image_id, anchor_idx)
    Output: int in [0, 2**32) - identical in every Python process, forever.
    """
    key = "|".join(str(p) for p in parts)
    digest = hashlib.sha256(key.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2 ** 32)


def select_exemplar_indices(n_gt: int,
                            anchor_idx: int,
                            n_exemplars: int,
                            image_id: str,
                            experiment_name: str = EXPERIMENT_NAME) -> list:
    """
    Choose which GT instances of ONE image are used as visual prompts.

    Input : n_gt        - number of GT boxes in the image
            anchor_idx  - index of the GT box this run is "about" (always a prompt)
            n_exemplars - how many prompts in total (1 or 3)
            image_id    - "<folder>/<image name>"
    Output: list of GT indices, ANCHOR FIRST, then the randomly sampled others.

    The anchor is always included; the remaining (n_exemplars - 1) slots are
    filled by sampling without replacement from the other GT boxes of the SAME
    image. If the image does not contain enough other boxes, fewer prompts are
    used (no duplication, no crash).
    """
    seed = stable_seed(experiment_name, image_id, anchor_idx)
    rng = np.random.default_rng(seed)

    others = [i for i in range(n_gt) if i != anchor_idx]
    n_needed = min(n_exemplars - 1, len(others))
    if n_needed > 0:
        chosen = [int(i) for i in rng.choice(others, size=n_needed, replace=False)]
    else:
        chosen = []
    return [int(anchor_idx)] + chosen


def format_prompt_id(exemplar_indices: list) -> str:
    """
    Human-readable id of a prompt set.
      single   -> "5"
      multiple -> "5+12+3"  (the ANCHOR is always the first number)
    """
    return "+".join(str(int(i)) for i in exemplar_indices)


# --- quick self-test: the same key must always give the same seed -------------
_demo = stable_seed(EXPERIMENT_NAME, "folderA/img_001", 3)
print("stable_seed demo :", _demo, "(identical in every session / machine)")
print("exemplar demo    :", select_exemplar_indices(10, 3, N_EXEMPLARS, "folderA/img_001"))

In [ ]:
# =============================================================================
# CELL 7 - DATASET AND YOLO ANNOTATION HELPERS
# =============================================================================
# Unchanged logic from the original notebook (it worked); only wrapped into
# functions and given comments.
# =============================================================================

def load_yolo_boxes(label_path, img_width, img_height, class_id=RUMEX_CLASS_ID):
    """
    Read a YOLO .txt annotation file and convert it to pixel corner boxes.

    Input : label_path            - YOLO txt file
            img_width, img_height - size of the ORIGINAL image in pixels
            class_id              - keep only this class (0 = Rumex)
    Output: np.ndarray (N, 4) float32, boxes as [x1, y1, x2, y2] in pixels.

    YOLO stores normalised (class, x_center, y_center, width, height).
    """
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue                      # skip empty lines
            if int(parts[0]) != class_id:
                continue                      # keep only the requested class
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc = xc * img_width, yc * img_height
            bw, bh = bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def find_label_path(image_filename_no_ext, folder_name):
    """
    Locate the YOLO txt belonging to an image, supporting both a mirrored folder
    structure (labels/<folder>/<name>.txt) and a flat one (labels/<name>.txt).
    Returns the path or None.
    """
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None


def discover_images():
    """
    Walk IMAGES_ROOT and collect every image together with its label file.
    Output: list of (folder, image_path, label_path, image_id) tuples,
            where image_id = "<folder>/<image name without extension>".
    """
    records = []
    for folder in sorted(os.listdir(IMAGES_ROOT)):
        folder_path = os.path.join(IMAGES_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(VALID_IMAGE_EXTENSIONS):
                continue
            name_no_ext = os.path.splitext(fname)[0]
            label_path = find_label_path(name_no_ext, folder)
            image_id = f"{folder}/{name_no_ext}"
            records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
    return records


def safe_filename(image_id: str) -> str:
    """'folder/name' -> 'folder__name' so it can be used inside a file name."""
    return image_id.replace("/", "__").replace(os.sep, "__")


def safe_crop(image, box, min_size=2):
    """
    Crop an exemplar from the full image, clamped to the image borders and to a
    minimum size. Guards against degenerate/out-of-range YOLO boxes, which would
    otherwise produce a 0-pixel crop and crash the strip composition.
    """
    x1, y1, x2, y2 = [int(round(float(v))) for v in box]
    x1 = max(0, min(x1, image.width - min_size))
    y1 = max(0, min(y1, image.height - min_size))
    x2 = min(image.width, max(x2, x1 + min_size))
    y2 = min(image.height, max(y2, y1 + min_size))
    return image.crop((x1, y1, x2, y2))


image_records = discover_images()
missing_labels = [r for r in image_records if r[2] is None]
valid_images = [r for r in image_records if r[2] is not None]

print(f"Discovered {len(image_records)} images in "
      f"{len(set(r[0] for r in image_records))} folders.")
print(f"With labels: {len(valid_images)} | without labels (skipped): {len(missing_labels)}")
if missing_labels:
    print("  first missing:", [r[3] for r in missing_labels[:5]])

In [ ]:
# =============================================================================
# CELL 9 - IoU
# =============================================================================
# Unchanged from the original notebook - it was already correct and vectorised.
# =============================================================================

def compute_iou_matrix(boxes1, boxes2):
    """
    Pairwise IoU between two sets of [x1, y1, x2, y2] boxes.

    Input : boxes1 (N, 4), boxes2 (M, 4)
    Output: (N, M) matrix, entry [i, j] = IoU(boxes1[i], boxes2[j])
    """
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])   # left
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])   # top
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])   # right
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])   # bottom

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

# How much of this predicted bounding box lies inside a certain tile or region?
# how many pixels of a bounding box overlap with another rectangular region
# only intersection area not oc=ver union
def intersection_area(box, region):
    """Plain intersection AREA (not IoU) between one box and one region."""
    x1 = max(box[0], region[0]); y1 = max(box[1], region[1]) # top and left of inter
    x2 = min(box[2], region[2]); y2 = min(box[3], region[3]) # bottom and riht of inter
    return max(0.0, x2 - x1) * max(0.0, y2 - y1) # area of inter = inter width x inter height


print("IoU helpers ready.")

In [ ]:
# =============================================================================
# CELL 10 - CORRECT ONE-TO-ONE MATCHING
# =============================================================================
# THE BUG IN THE OLD CODE
# The old matcher did:
#     best_gt = argmax(IoU[pred])          <- global best, ignoring availability
#     if best_iou >= thr and best_gt free: match
# So if the globally best GT was already taken, the prediction was thrown away
# even when ANOTHER, still free GT had IoU >= threshold. That under-counted TPs
# (recall too low, false positives too high) on images with clustered plants.
#
# THE CORRECT RULE (implemented here)
#   sort predictions by confidence, high -> low
#   for each prediction:
#       look ONLY at GT boxes that are still unmatched
#       take the unmatched GT with the highest IoU
#       if that IoU >= evaluation IoU threshold -> match, else leave unmatched
#   one GT can be matched by at most one prediction, and vice versa.
#
# This single function is used for TP / FP / FN / precision / recall / F1 /
# IoU1 / IoU2 / confusion matrices, so the whole thesis uses one definition.
# =============================================================================

def match_one_to_one(pred_boxes, pred_scores, gt_boxes, iou_threshold=EVAL_IOU_THRESHOLD):
    """
    Input : pred_boxes  (P, 4), pred_scores (P,), gt_boxes (G, 4), iou_threshold
    Output: dict with
        'pred_match_gt' (P,) int   - matched GT index per prediction, -1 if unmatched
        'gt_match_pred' (G,) int   - matched prediction index per GT,  -1 if unmatched
        'pred_iou'      (P,) float - IoU of the accepted match, 0.0 if unmatched
        'matched_ious'  list       - IoU values of all accepted matches
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)

    pred_match_gt = np.full(n_pred, -1, dtype=np.int64)
    gt_match_pred = np.full(n_gt, -1, dtype=np.int64)
    pred_iou = np.zeros(n_pred, dtype=np.float32) # stores the IoU of the accepted match for every prediction

    if n_pred == 0 or n_gt == 0:
        return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
                "pred_iou": pred_iou, "matched_ious": []}

    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    gt_free = np.ones(n_gt, dtype=bool) # Track which GTs are still available

    # 'stable' keeps the original order for equal scores -> fully deterministic.
    order = np.argsort(-np.asarray(pred_scores, dtype=np.float32), kind="stable") # Sort predictions by confidence

    matched_ious = []
    for p in order:
        if not gt_free.any():
            break                                   # nothing left to match, every GT already has a prediction
        # Consider ONLY currently unmatched GT boxes (this is the fix).
        candidate_ious = np.where(gt_free, iou[p], -1.0)
        g = int(np.argmax(candidate_ious))  # Select the best FREE GT
        if candidate_ious[g] >= iou_threshold:
            gt_free[g] = False
            pred_match_gt[p] = g
            gt_match_pred[g] = p
            pred_iou[p] = candidate_ious[g]
            matched_ious.append(float(candidate_ious[g]))

    return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
            "pred_iou": pred_iou, "matched_ious": matched_ious}


def safe_f1(precision, recall):
    """F1 = 2PR/(P+R) with a safe zero denominator (returns 0.0)."""
    denom = precision + recall
    return float(2.0 * precision * recall / denom) if denom > 0 else 0.0


print("One-to-one matcher ready (unmatched-GT-aware).")

In [ ]:
# =============================================================================
# CELL 11 - TILES: GENERATED ONCE PER IMAGE, REUSED BY EVERY ANCHOR
# =============================================================================
# The old code re-cropped every tile for every anchor. Because a single UAV image
# can have dozens of anchors, that repeated the (expensive) cropping work dozens
# of times. Now: open the image once -> build the tile list once -> reuse.
# Tiles are kept as CPU/PIL images (never on the GPU).
# =============================================================================

def tile_bboxes(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    """
    Sliding-window tiles covering the whole image with overlap, so a plant lying
    on a tile border is fully visible in at least one window.
    Output: list of (x1, y1, x2, y2) in ORIGINAL image coordinates.
    """
    step = max(1, tile_size - overlap)
    tiles = []
    for y in range(0, img_h, step):
        for x in range(0, img_w, step):
            x2 = min(x + tile_size, img_w)
            y2 = min(y + tile_size, img_h)
            x1 = max(0, x2 - tile_size)
            y1 = max(0, y2 - tile_size)
            tiles.append((x1, y1, x2, y2))
    return list(dict.fromkeys(tiles))   # remove duplicates, keep order


def build_tile_cache(image, use_tiling=USE_TILING,
                     tile_size=TILE_SIZE, overlap=OVERLAP,
                     cache_in_memory=CACHE_TILES_IN_MEMORY):
    """
    Build the tile list for ONE already-open image.

    Input : image - PIL image of the full UAV photo
    Output: list of dicts, one per tile:
            {'tile_id', 'x1', 'y1', 'x2', 'y2', 'image' (PIL or None)}
            'image' is None when cache_in_memory=False (cropped on demand).

    If use_tiling is False the whole image is returned as a single "tile", which
    keeps the rest of the pipeline identical for the no-tiling experiment.
    """
    if use_tiling:
        coords = tile_bboxes(image.width, image.height, tile_size, overlap)
    else:
        coords = [(0, 0, image.width, image.height)]

    cache = []
    for tid, (x1, y1, x2, y2) in enumerate(coords):
        cache.append({
            "tile_id": tid, "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "image": image.crop((x1, y1, x2, y2)) if cache_in_memory else None,
        })
    return cache


def get_tile_image(tile, full_image):
    """Return the tile's PIL image, cropping on demand if it was not cached."""
    if tile["image"] is not None:
        return tile["image"]
    return full_image.crop((tile["x1"], tile["y1"], tile["x2"], tile["y2"]))


print(f"Tiling helpers ready (TILE_SIZE={TILE_SIZE}, OVERLAP={OVERLAP}).")

In [ ]:
# =============================================================================
# CELL 12 - LOCAL BACKGROUND PATCH  (bug fixed)
# =============================================================================
# WHY THE STRIP HAS A TEXTURED BACKGROUND
# The exemplar crops are pasted into a strip above the tile. A flat white strip
# would create a hard artificial edge that the image encoder's self-attention can
# latch onto, and the exemplar feature would describe "a plant in a void" instead
# of "a plant in grass". Sampling real texture from the same tile keeps colour,
# brightness and grain consistent with the current lighting conditions.
#
# THE BUG
# The old version cropped the TOP-LEFT corner of the tile - exactly the rows that
# are then pasted directly ABOVE the tile. The same texture therefore appeared
# twice, immediately adjacent to itself: an obvious repeating seam.
#
# THE FIX
#   - sample from a RANDOM offset inside the tile, deliberately skipping the top
#     band that would sit next to its own copy
#   - the offset comes from a deterministic RNG (stable_seed) -> reproducible
#   - apply a light Gaussian blur so leftover structure does not look like a
#     second, sharp copy of real plants
# =============================================================================

def get_local_background_patch(tile_img, patch_w, patch_h, rng,
                               blur_radius=BACKGROUND_BLUR_RADIUS):
    """
    Input : tile_img        - PIL image of the current tile
            patch_w/patch_h - required size of the strip background
            rng             - np.random.Generator (deterministically seeded)
    Output: PIL image of size (patch_w, patch_h) with local-looking texture.
    """
    tw, th = tile_img.size
    crop_w = min(patch_w, tw)
    crop_h = min(patch_h, th)

    max_x0 = tw - crop_w
    max_y0 = th - crop_h
    # Skip the first patch_h rows: they are the ones that end up directly below
    # the strip, so re-using them would create the adjacent duplication.
    min_y0 = min(patch_h, max_y0)

    x0 = int(rng.integers(0, max_x0 + 1)) if max_x0 > 0 else 0
    y0 = int(rng.integers(min_y0, max_y0 + 1)) if max_y0 > min_y0 else max_y0

    patch = tile_img.crop((x0, y0, x0 + crop_w, y0 + crop_h))
    if patch.size != (patch_w, patch_h):
        patch = patch.resize((patch_w, patch_h), Image.BILINEAR)
    if blur_radius and blur_radius > 0:
        patch = patch.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    return patch


def make_feather_mask(size, feather_width=FEATHER_WIDTH):
    """
    Soft alpha mask so a pasted exemplar crop blends into the strip background
    instead of showing a hard rectangular border. Unchanged from the original.

    Input : size (w, h) of the crop, feather_width in px
    Output: PIL 'L' image used as the paste mask
    """
    w, h = size
    mask = np.full((h, w), 255.0, dtype=np.float32)     # start fully opaque
    effective = min(feather_width, h // 2, w // 2)
    if effective >= 1:
        for i in range(effective):
            alpha = 255.0 * (i + 1) / effective
            mask[i, :] = np.minimum(mask[i, :], alpha)                   # top
            mask[h - 1 - i, :] = np.minimum(mask[h - 1 - i, :], alpha)   # bottom
            mask[:, i] = np.minimum(mask[:, i], alpha)                   # left
            mask[:, w - 1 - i] = np.minimum(mask[:, w - 1 - i], alpha)   # right
    return Image.fromarray(mask.astype(np.uint8), mode="L")


print("Background sampling + feather mask ready.")

In [ ]:
# =============================================================================
# CELL 13 - COMPOSE  (exemplar strip on top + real tile below)  (bug fixed)
# =============================================================================
# THE BUG
# The old code computed
#     canvas_w = max(tile_w, strip_content_w)
# With 3 large exemplars the strip was wider than the tile, so the canvas was
# wider than the tile. Image.new("RGB", ...) fills new pixels with BLACK, and the
# area to the RIGHT of the pasted tile stayed black -> a large artificial black
# rectangle inside the model input.
#
# THE FIX (simple and robust)
#   canvas width is ALWAYS exactly the tile width, so nothing can remain unpainted
#   next to the tile. If the exemplars do not fit in that width, ALL crops are
#   scaled down by ONE common factor:
#       scale = available_width / total_crop_width
#   Using a single common factor preserves each crop's aspect ratio AND the
#   relative size differences between the exemplars. The UAV tile itself is never
#   resized or distorted. Works for 1 and for 3 exemplars.
# =============================================================================

def compose_tile_with_exemplars(tile_img, crop_images, rng,
                                margin=STRIP_MARGIN,
                                feather_width=FEATHER_WIDTH,
                                max_strip_height_fraction=MAX_STRIP_HEIGHT_FRACTION):
    """
    Input : tile_img    - PIL image of the tile
            crop_images - list of PIL exemplar crops (1 or 3)
            rng         - deterministic np.random.Generator for the background
    Output: composed  - PIL image actually sent to SAM3
            crop_boxes- list of [x1,y1,x2,y2] of each exemplar in COMPOSED coords
                        (these are the positive visual prompts)
            offset    - (dx, dy) where the real tile starts inside 'composed'
    """
    n = len(crop_images)
    canvas_w = tile_img.width                       # never wider than the tile
    available_w = canvas_w - margin * (n + 1)       # space left for the crops
    total_crop_w = sum(c.width for c in crop_images)

    # --- 1) shrink the exemplars (aspect ratio preserved) if they do not fit ---
    scale = 1.0
    if total_crop_w > 0 and available_w > 0 and total_crop_w > available_w:
        scale = available_w / float(total_crop_w)

    # --- 2) optional additional height clamp (disabled by default) -------------
    if max_strip_height_fraction is not None:
        max_crop_h = max(1.0, max_strip_height_fraction * tile_img.height - 2 * margin)
        tallest = max(c.height for c in crop_images)
        if tallest * scale > max_crop_h:
            scale = min(scale, max_crop_h / float(tallest))

    if scale < 1.0:
        crop_images = [
            c.resize((max(1, int(round(c.width * scale))),
                      max(1, int(round(c.height * scale)))), Image.BILINEAR)
            for c in crop_images
        ]

    strip_h = max(c.height for c in crop_images) + 2 * margin
    canvas_h = strip_h + tile_img.height

    # --- 3) paint the whole strip with local texture (no unpainted pixels) -----
    composed = Image.new("RGB", (canvas_w, canvas_h))
    composed.paste(get_local_background_patch(tile_img, canvas_w, strip_h, rng), (0, 0))

    # --- 4) paste the real tile below the strip; it fills the full canvas width -
    offset = (0, strip_h)
    composed.paste(tile_img, offset)

    # --- 5) paste the exemplars side by side, with feathered edges -------------
    crop_boxes = []
    cursor_x = margin
    for crop in crop_images:
        composed.paste(crop, (cursor_x, margin), make_feather_mask(crop.size, feather_width))
        crop_boxes.append([cursor_x, margin, cursor_x + crop.width, margin + crop.height])
        cursor_x += crop.width + margin

    return composed, crop_boxes, offset


print("Exemplar strip composition ready (canvas width == tile width, no black padding).")

In [ ]:
# =============================================================================
# CELL 14 - TARGET-REGION FILTER (bug fixed) + PLAUSIBILITY FILTER
# =============================================================================
# THE BUG IN keep_only_target_region_detections
# The old rule was "keep the box only if y1 >= dy - 5", i.e. it looked ONLY at the
# top edge of the box. A real Rumex plant sitting at the very top of a tile whose
# predicted box leaks a few dozen pixels into the exemplar strip was deleted, even
# though almost all of the box was inside the tile.
#
# THE FIX - explicit geometric criterion
# The composed image contains two regions:
#     strip region : y in [0, dy)
#     tile  region : y in [dy, dy + tile_h)
# A detection is kept if at least TILE_REGION_MIN_FRACTION (= 0.50) of its AREA
# lies inside the TILE region, i.e. the box belongs to whichever region holds the
# majority of it. Kept boxes are then CLIPPED to the tile region and shifted into
# tile coordinates. No other heuristic is applied here.
# =============================================================================

def keep_only_target_region_detections(boxes, scores, fill_ratios,
                                       offset, tile_w, tile_h,
                                       min_fraction_inside=TILE_REGION_MIN_FRACTION):
    """
    Input : boxes (N,4) in COMPOSED coordinates, scores (N,), fill_ratios (N,)
            offset (dx, dy) = where the tile starts inside the composed image
            tile_w, tile_h  = size of the real tile
    Output: boxes in TILE coordinates, scores, fill_ratios (all filtered)
    """
    dx, dy = offset
    tile_region = (dx, dy, dx + tile_w, dy + tile_h)

    kept_boxes, kept_scores, kept_fills = [], [], []
    for box, score, fill in zip(boxes, scores, fill_ratios):
        x1, y1, x2, y2 = [float(v) for v in box]
        box_area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
        if box_area <= 0:
            continue
        # fraction of the predicted box that lies inside the real tile
        fraction_inside = intersection_area((x1, y1, x2, y2), tile_region) / box_area
        if fraction_inside < min_fraction_inside:
            continue                                   # belongs to the strip
        # clip to the tile region, then convert composed -> tile coordinates
        cx1 = min(max(x1, tile_region[0]), tile_region[2]) - dx
        cy1 = min(max(y1, tile_region[1]), tile_region[3]) - dy
        cx2 = min(max(x2, tile_region[0]), tile_region[2]) - dx
        cy2 = min(max(y2, tile_region[1]), tile_region[3]) - dy
        kept_boxes.append([cx1, cy1, cx2, cy2])
        kept_scores.append(float(score))
        kept_fills.append(float(fill))
    return kept_boxes, kept_scores, kept_fills


def filter_implausible_boxes(boxes, scores, fill_ratios, tile_w, tile_h,
                             min_fill_ratio=MIN_FILL_RATIO,
                             max_area_fraction=MAX_AREA_FRACTION,
                             edge_margin=EDGE_MARGIN):
    """
    Remove detections that cannot be a single Rumex plant. SAME thresholds and
    SAME logic as the original notebook; the only change is that the mask-fill
    ratio is received as a PRE-COMPUTED number instead of a stored mask, because
    masks must never be kept (KEEP_MASKS = False).

    Input : boxes in TILE coordinates, scores, fill_ratios, tile size
    Output: the surviving boxes / scores / fill_ratios
    """
    kept_boxes, kept_scores, kept_fills = [], [], []
    tile_area = float(tile_w * tile_h)
    for box, score, fill in zip(boxes, scores, fill_ratios):
        x1, y1, x2, y2 = box
        bw, bh = x2 - x1, y2 - y1
        if bw <= edge_margin or bh <= edge_margin:          # tiny boxes
            continue
        if (bw * bh) / tile_area > max_area_fraction:       # absurdly large boxes
            continue
        if fill < min_fill_ratio:                           # hollow boxes
            continue
        kept_boxes.append(box)
        kept_scores.append(score)
        kept_fills.append(fill)
    return kept_boxes, kept_scores, kept_fills


print("Target-region filter (>=50% area inside tile) + plausibility filter ready.")

In [ ]:
# =============================================================================
# CELL 15 - SAM3 BATCH INFERENCE  (FP16, no masks kept)
# Same batched, FP16, no-masks-kept inference as E02_2
# =============================================================================

def _to_numpy(x):
    """Torch tensor (any device/dtype) or numpy array -> float32 numpy array."""
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x, dtype=np.float32)


def _fill_ratios_from_masks(boxes_np, masks, binarise_at=0.5):
    """
    Fraction of each predicted box that is actually covered by its mask - the
    single number the plausibility filter needs. Only the small box region is
    materialised; the mask itself is dropped by the caller immediately after.
    """
    n = len(boxes_np)
    fills = np.zeros(n, dtype=np.float32)
    if masks is None or n == 0:
        return fills
    for i in range(n):
        m = masks[i]
        h, w = int(m.shape[-2]), int(m.shape[-1])
        x1, y1, x2, y2 = boxes_np[i]
        x1c, y1c = int(max(0, np.floor(x1))), int(max(0, np.floor(y1)))
        x2c, y2c = int(min(w, np.ceil(x2))), int(min(h, np.ceil(y2)))
        if x2c <= x1c or y2c <= y1c:
            continue
        region = m[..., y1c:y2c, x1c:x2c]
        if torch.is_tensor(region):
            if region.dtype == torch.bool:
                fills[i] = float(region.float().mean())
            else:
                fills[i] = float((region > binarise_at).float().mean())
        else:
            region = np.asarray(region)
            fills[i] = float((region > binarise_at).mean()) if region.size else 0.0
    return fills


def sam3_infer_batch(composed_images, crop_boxes_batch,
                     threshold=SAM3_INFERENCE_THRESHOLD):
    """
    Run SAM3 on a batch of composed images (exemplar strip + tile).

    Output: list (same length as composed_images) of (boxes, scores, fill_ratios),
            all numpy, boxes in COMPOSED-image coordinates, every detection with
            score >= threshold. Masks are NOT returned.
    """
    inputs = sam3_processor(
        images=composed_images,
        input_boxes=[[[float(v) for v in b] for b in boxes] for boxes in crop_boxes_batch],
        input_boxes_labels=[[1] * len(boxes) for boxes in crop_boxes_batch],  # 1 = positive
        return_tensors="pt",
    ).to(sam3_model.device)

    if MODEL_DTYPE == torch.float16 and "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    per_image = []
    with torch.inference_mode():
        if MODEL_DTYPE == torch.float16:
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = sam3_model(**inputs)
        else:
            outputs = sam3_model(**inputs)

        # Post-process at the SAM3_INFERENCE_THRESHOLD floor first...
        results = sam3_processor.post_process_instance_segmentation(
            outputs,
            threshold=threshold,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=inputs.get("original_sizes").tolist(),
        )

        for i, res in enumerate(results):
            boxes = _to_numpy(res["boxes"]).reshape(-1, 4)
            scores = _to_numpy(res["scores"]).reshape(-1)
            fills = _fill_ratios_from_masks(boxes, res.get("masks", None))
            if "masks" in res:
                res["masks"] = None

            per_image.append((boxes, scores, fills))

    del inputs, outputs, results
    return per_image


print("SAM3 batched inference helpers ready.")

In [ ]:
# =============================================================================
# CELL 16 - OPTIONAL GLOBAL-CONTEXT PASS (opt-in, ADD_GLOBAL_CONTEXT_PASS)
# =============================================================================
# An EXTRA single pass over the whole image, downscaled by GLOBAL_DOWNSCALE,
# run through the exact same exemplar-strip + SAM3 + filter pipeline as a tile
# (it IS treated as one big "tile" -- no cropping). Its detections are rescaled
# back to full-resolution coordinates and returned so the caller (CELL 17) can
# simply append them to the tiled pre-NMS detection pool. Because they join the
# SAME pool that gets NMS'd and evaluated offline, nothing downstream needs any
# special-casing for this pass.
# =============================================================================

def run_global_context_pass(image, exemplar_crops, run_seed,
                            downscale=GLOBAL_DOWNSCALE,
                            threshold=SAM3_INFERENCE_THRESHOLD):
    """
    Input : image          - the open, full-resolution PIL image
            exemplar_crops - list of PIL exemplar crops (same ones used for tiles)
            run_seed       - deterministic seed for the strip background sampling
    Output: dict with 'boxes' (N,4), 'scores' (N,), 'fill_ratio' (N,) in
            ORIGINAL-IMAGE coordinates. Empty arrays if nothing is detected.
    """
    small_w = max(1, image.width // downscale)
    small_h = max(1, image.height // downscale)
    small_img = image.resize((small_w, small_h), Image.BILINEAR)

    rng = np.random.default_rng(stable_seed(run_seed, "global_pass_bg"))
    composed, crop_boxes, offset = compose_tile_with_exemplars(small_img, exemplar_crops, rng)

    (boxes, scores, fills), = sam3_infer_batch([composed], [crop_boxes], threshold)

    boxes, scores, fills = keep_only_target_region_detections(
        boxes, scores, fills, offset, small_img.width, small_img.height)
    boxes, scores, fills = filter_implausible_boxes(
        boxes, scores, fills, small_img.width, small_img.height)

    # small_img coords -> original image coords
    scale = image.width / small_img.width
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4) * scale

    composed.close()
    small_img.close()

    return {
        "boxes": boxes.reshape(-1, 4),
        "scores": np.asarray(scores, dtype=np.float32).reshape(-1),
        "fill_ratio": np.asarray(fills, dtype=np.float32).reshape(-1),
    }


print("Global-context-pass helper ready. Enabled:", ADD_GLOBAL_CONTEXT_PASS,
      f"(downscale={GLOBAL_DOWNSCALE})" if ADD_GLOBAL_CONTEXT_PASS else "")


In [ ]:
# =============================================================================
# CELL 17 - RUN ONE ANCHOR OVER ALL TILES (+ optional global-context pass)
# =============================================================================
# Runs SAM3 for ONE anchor/exemplar configuration over ALL cached tiles (same
# as E02_2), and -- if ADD_GLOBAL_CONTEXT_PASS is on -- ALSO runs the extra
# downsampled whole-image pass from CELL 16 and appends its detections to the
# SAME pre-NMS pool, tagged with tile_id=-1 so they're identifiable later
# (e.g. for diagnostics) without needing a separate storage schema.
# =============================================================================

def run_anchor_over_tiles(tile_cache, full_image, exemplar_crops, run_seed,
                          batch_size=BATCH_SIZE, threshold=SAM3_INFERENCE_THRESHOLD,
                          add_global_pass=ADD_GLOBAL_CONTEXT_PASS):
    """
    Output: dict of numpy arrays, all in ORIGINAL-IMAGE coordinates:
            boxes (N,4), scores (N,), fill_ratio (N,), tile_id (N,),
            tile_boxes (N,4)  <- the tile each detection came from
                                 (tile_id == -1, tile_boxes == whole image
                                 extent, for detections from the global pass)
    These are the PRE-NMS detections (score >= SAM3_INFERENCE_THRESHOLD, no NMS)
    that get written to disk for the offline evaluation.
    """
    all_boxes, all_scores, all_fills, all_tids, all_tboxes = [], [], [], [], []

    for start in range(0, len(tile_cache), batch_size):
        batch_tiles = tile_cache[start:start + batch_size]

        composed_images, crop_boxes_batch, offsets, sizes = [], [], [], []
        for tile in batch_tiles:
            tile_img = get_tile_image(tile, full_image)
            rng = np.random.default_rng(stable_seed(run_seed, "tile", tile["tile_id"]))
            composed, crop_boxes, offset = compose_tile_with_exemplars(
                tile_img, exemplar_crops, rng)
            composed_images.append(composed)
            crop_boxes_batch.append(crop_boxes)
            offsets.append(offset)
            sizes.append((tile_img.width, tile_img.height))

        batch_results = sam3_infer_batch(composed_images, crop_boxes_batch, threshold)

        for tile, (boxes, scores, fills), offset, (tw, th) in zip(
                batch_tiles, batch_results, offsets, sizes):
            boxes, scores, fills = keep_only_target_region_detections(
                boxes, scores, fills, offset, tw, th)
            boxes, scores, fills = filter_implausible_boxes(boxes, scores, fills, tw, th)
            for b, s, f in zip(boxes, scores, fills):
                all_boxes.append([b[0] + tile["x1"], b[1] + tile["y1"],
                                  b[2] + tile["x1"], b[3] + tile["y1"]])
                all_scores.append(float(s))
                all_fills.append(float(f))
                all_tids.append(int(tile["tile_id"]))
                all_tboxes.append([tile["x1"], tile["y1"], tile["x2"], tile["y2"]])

        for ct in composed_images:
            ct.close()
        del composed_images, crop_boxes_batch, batch_results

    used_global_pass = False
    if add_global_pass:
        global_det = run_global_context_pass(full_image, exemplar_crops, run_seed)
        n_global = len(global_det["scores"])
        used_global_pass = n_global > 0
        for i in range(n_global):
            b = global_det["boxes"][i]
            all_boxes.append([float(b[0]), float(b[1]), float(b[2]), float(b[3])])
            all_scores.append(float(global_det["scores"][i]))
            all_fills.append(float(global_det["fill_ratio"][i]))
            all_tids.append(-1)  # sentinel: global-context-pass detection
            all_tboxes.append([0, 0, full_image.width, full_image.height])

    return {
        "boxes": np.array(all_boxes, dtype=np.float32).reshape(-1, 4),
        "scores": np.array(all_scores, dtype=np.float32).reshape(-1),
        "fill_ratio": np.array(all_fills, dtype=np.float32).reshape(-1),
        "tile_id": np.array(all_tids, dtype=np.int32).reshape(-1),
        "tile_boxes": np.array(all_tboxes, dtype=np.int32).reshape(-1, 4),
    }, used_global_pass


print("Anchor-over-tiles (+ optional global pass) helper ready.")


In [ ]:
# =============================================================================
# CELL 18 - PRE-NMS DETECTION STORAGE
# =============================================================================
# WHAT IS SAVED AND WHY
# For every run (= one image x one anchor) we store the detections AFTER
#   SAM3 inference at 0.30 -> target-region filtering -> plausibility filtering
#   -> conversion to original-image coordinates
# but BEFORE
#   any operating confidence threshold and BEFORE NMS.
#
# That is exactly the state needed to replay any (confidence, NMS IoU) pair
# offline without ever running SAM3 again. Masks are never stored.
# NPZ is used because it is compact and loads fast; the metric tables are CSV.
# =============================================================================

def run_npz_path(image_id, anchor_idx):
    """Path of the NPZ holding the pre-NMS detections of one run."""
    return os.path.join(RAW_DETECTIONS_DIR,
                        f"{safe_filename(image_id)}__anchor{int(anchor_idx):03d}.npz")


def save_run_detections(image_id, anchor_idx, detections, gt_boxes,
                        prompt_indices, image_size):
    """
    Write one run's pre-NMS detections to NPZ. The file is self-contained: it also
    stores the GT boxes and the prompt indices, so the whole offline evaluation
    can run without re-opening images or label files.
    """
    path = run_npz_path(image_id, anchor_idx)
    np.savez_compressed(
        path,
        experiment_name=np.array(EXPERIMENT_NAME),
        image_ID=np.array(image_id),
        anchor_idx=np.array(int(anchor_idx)),
        prompt_indices=np.array(prompt_indices, dtype=np.int32),
        image_width=np.array(int(image_size[0])),
        image_height=np.array(int(image_size[1])),
        gt_boxes=gt_boxes.astype(np.float32),
        boxes=detections["boxes"].astype(np.float32),        # x1,y1,x2,y2 (original img)
        scores=detections["scores"].astype(np.float32),      # confidence >= 0.30
        fill_ratio=detections["fill_ratio"].astype(np.float32),
        tile_id=detections["tile_id"].astype(np.int32),      # which tile produced it
        tile_boxes=detections["tile_boxes"].astype(np.int32),# that tile's extent
    )
    return path


def load_run_detections(path):
    """Read one run NPZ back into a plain python dict."""
    with np.load(path, allow_pickle=False) as z:
        return {
            "image_ID": str(z["image_ID"]),
            "anchor_idx": int(z["anchor_idx"]),
            "prompt_indices": z["prompt_indices"].astype(int),
            "image_width": int(z["image_width"]),
            "image_height": int(z["image_height"]),
            "gt_boxes": z["gt_boxes"].reshape(-1, 4),
            "boxes": z["boxes"].reshape(-1, 4),
            "scores": z["scores"].reshape(-1),
            "fill_ratio": z["fill_ratio"].reshape(-1),
            "tile_id": z["tile_id"].reshape(-1),
            "tile_boxes": z["tile_boxes"].reshape(-1, 4),
        }


print("Pre-NMS detection storage ready ->", RAW_DETECTIONS_DIR)

In [ ]:
# =============================================================================
# CELL 19 - MAIN GPU INFERENCE LOOP
# =============================================================================
# Same structure as E02_2: open image once, build tiles once, loop anchors,
# save PRE-NMS detections. NO confidence sweep, NO NMS sweep, NO metric
# computation here - all done offline in the following cells. Resumable via
# runs_manifest.csv.
#
# ADDED: a lightweight RAM guard (MEM_STOP_THRESHOLD_PCT, CELL 3), carried over
# from this notebook's own code -- tiling + the global pass use more host RAM
# than a single-pass whole-image approach. On breach it stops CLEANLY (not a
# hard crash); rerunning this cell resumes from the manifest.
# =============================================================================

done_runs = set()
manifest_exists = os.path.exists(RUN_MANIFEST_CSV)
if manifest_exists:
    _m = pd.read_csv(RUN_MANIFEST_CSV)
    _m = _m[_m["experiment_name"] == EXPERIMENT_NAME]
    done_runs = set(zip(_m["image_ID"].astype(str), _m["anchor_idx"].astype(int)))
    print(f"Resuming: {len(done_runs)} runs already finished for {EXPERIMENT_NAME}.")

manifest_file = open(RUN_MANIFEST_CSV, "a", newline="")
manifest_writer = csv.DictWriter(manifest_file, fieldnames=MANIFEST_COLUMNS)
if not manifest_exists:
    manifest_writer.writeheader()

start_time = time.time()
n_new_runs = 0
image_times = []
n_total_images = len(valid_images)

for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
    mem_pct = psutil.virtual_memory().percent
    if mem_pct > MEM_STOP_THRESHOLD_PCT:
        print(f"RAM at {mem_pct:.0f}% (threshold {MEM_STOP_THRESHOLD_PCT}%) -- "
              f"stopping cleanly before {image_id} to avoid a hard crash. "
              f"Rerun this cell to resume from where you left off.")
        break

    image_t0 = time.time()

    # ---------------- open the original image exactly once --------------------
    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size
    gt_boxes = load_yolo_boxes(label_path, img_w, img_h, RUMEX_CLASS_ID)
    n_gt = len(gt_boxes)

    if n_gt == 0:
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
        image.close(); del image; gc.collect()
        continue

    if all((image_id, a) in done_runs for a in range(n_gt)):
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: all "
              f"{n_gt} anchors already done, skipped.")
        image.close(); del image; gc.collect()
        continue

    # ---------------- build the tiles exactly once ----------------------------
    tile_cache = build_tile_cache(image)
    n_tiles = len(tile_cache)

    for anchor_idx in range(n_gt):                 # every GT box is an anchor once
        if (image_id, anchor_idx) in done_runs:
            continue

        mem_pct = psutil.virtual_memory().percent
        if mem_pct > MEM_STOP_THRESHOLD_PCT:
            print(f"RAM at {mem_pct:.0f}% mid-image at {image_id} anchor={anchor_idx} -- "
                  f"stopping cleanly. Completed anchors are already saved; rerun to resume.")
            manifest_file.close()
            for t in tile_cache:
                t["image"] = None
            del tile_cache
            image.close(); del image; gc.collect()
            raise SystemExit("Stopped cleanly due to RAM threshold - rerun this cell to resume.")

        run_t0 = time.time()

        # deterministic prompt selection (SHA-256 based, see CELL 6)
        exemplar_indices = select_exemplar_indices(n_gt, anchor_idx, N_EXEMPLARS, image_id)
        prompt_id = format_prompt_id(exemplar_indices)
        exemplar_crops = [safe_crop(image, gt_boxes[i]) for i in exemplar_indices]

        run_seed = stable_seed(EXPERIMENT_NAME, image_id, anchor_idx, "strip_bg")
        detections, used_global_pass = run_anchor_over_tiles(
            tile_cache, image, exemplar_crops, run_seed)

        npz_path = save_run_detections(image_id, anchor_idx, detections,
                                       gt_boxes, exemplar_indices, (img_w, img_h))

        run_seconds = time.time() - run_t0
        manifest_writer.writerow({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": image_id,
            "anchor_idx": anchor_idx,
            "Prompt_ID": prompt_id,
            "Prompt_Type": PROMPT_TYPE,
            "n_gt": n_gt,
            "n_prompt_gt": len(exemplar_indices),
            "n_detections_pre_nms": int(len(detections["scores"])),
            "n_tiles": n_tiles,
            "used_global_pass": used_global_pass,
            "image_width": img_w,
            "image_height": img_h,
            "npz_file": os.path.basename(npz_path),
            "inference_seconds": round(run_seconds, 2),
        })
        manifest_file.flush()
        n_new_runs += 1

        print(f"  [{EXPERIMENT_NAME}] run #{n_new_runs} | {image_id} | "
              f"anchor={anchor_idx} ({anchor_idx + 1}/{n_gt}) | prompt={prompt_id} | "
              f"tiles={n_tiles} | global_pass={used_global_pass} | "
              f"pre-NMS detections={len(detections['scores'])} | {run_seconds:.1f}s")

        del detections, exemplar_crops
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    # ---------------- release the image and its tile cache --------------------
    for t in tile_cache:
        t["image"] = None
    del tile_cache
    image.close()
    del image
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    image_elapsed = time.time() - image_t0
    image_times.append(image_elapsed)
    avg_per_image = float(np.mean(image_times))
    eta = (n_total_images - img_idx) * avg_per_image
    rss_gb = psutil.Process().memory_info().rss / 1e9
    print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} done | "
          f"{n_gt} GT box(es) | {image_elapsed:.1f}s | avg/image={avg_per_image:.1f}s | "
          f"ETA={eta / 60:.1f} min ({eta / 3600:.2f} h) | [MEM] RSS={rss_gb:.2f} GB")

manifest_file.close()
total_elapsed = time.time() - start_time
print(f"\nInference finished for {EXPERIMENT_NAME}: {n_new_runs} new runs.")
print(f"Total time: {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.2f} h)")
print(f"Pre-NMS detections in: {RAW_DETECTIONS_DIR}")


In [ ]:
# =============================================================================
# CELL 20 - LOAD CACHED PRE-NMS DETECTIONS
# =============================================================================
# From here on SAM3 is never touched again. Everything below works on the NPZ
# files written in CELL 16, so you can restart the runtime, free the GPU and
# still redo the complete evaluation in a few minutes.
# =============================================================================

manifest = pd.read_csv(RUN_MANIFEST_CSV)
manifest = manifest[manifest["experiment_name"] == EXPERIMENT_NAME].copy()
manifest = manifest.drop_duplicates(subset=["image_ID", "anchor_idx"], keep="last")

runs = []
for _, row in manifest.iterrows():
    path = os.path.join(RAW_DETECTIONS_DIR, str(row["npz_file"]))
    if not os.path.exists(path):
        print("MISSING npz (skipped):", path)
        continue
    run = load_run_detections(path)
    run["Prompt_ID"] = str(row["Prompt_ID"])
    run["Prompt_Type"] = str(row["Prompt_Type"])
    runs.append(run)

print(f"Loaded {len(runs)} runs "
      f"({manifest['image_ID'].nunique()} images) for {EXPERIMENT_NAME}.")
print("Total pre-NMS detections:", int(sum(len(r['scores']) for r in runs)))
print("Total GT boxes over all runs:", int(sum(len(r['gt_boxes']) for r in runs)))

In [ ]:
# =============================================================================
# CELL 21 - OFFLINE NMS
# =============================================================================
# Because the same plant is visible in several overlapping tiles, it can be
# detected several times. NMS keeps the highest-scoring box of each overlapping
# group. The NMS IoU threshold is now a TUNABLE parameter applied offline, so the
# raw pre-NMS detections stay untouched on disk and every value can be replayed.
#
# 'Provenance' = we also record WHICH detection suppressed which. That lets the
# false-positive diagnostics (CELL 29) count how many duplicates came from a
# DIFFERENT tile (cross-tile duplication) versus the same tile.
# =============================================================================

def nms_with_provenance(boxes, scores, iou_threshold):
    """
    Input : boxes (N,4), scores (N,), iou_threshold
    Output: keep (list of kept indices, highest score first)
            suppressed (list of (suppressed_index, suppressor_index))
    A detection is suppressed when its IoU with an already kept, higher-scoring
    detection is GREATER than the threshold (same convention as the original code).
    """
    n = len(boxes)
    if n == 0:
        return [], []
    order = list(np.argsort(-np.asarray(scores, dtype=np.float32), kind="stable"))
    keep, suppressed = [], []
    while order:
        i = int(order[0])
        keep.append(i)
        rest = np.array(order[1:], dtype=int)
        if rest.size == 0:
            break
        ious = compute_iou_matrix(boxes[i:i + 1], boxes[rest])[0]
        for s in rest[ious > iou_threshold]:
            suppressed.append((int(s), i))
        order = list(rest[ious <= iou_threshold])
    return keep, suppressed


def apply_nms_to_run(run, iou_threshold):
    """
    Apply NMS to one run's pre-NMS detections.

    Output: dict with 'boxes', 'scores', 'tile_id', 'tile_boxes' of the surviving
            detections SORTED BY SCORE (high -> low), plus the duplicate-source
            counters used later by the diagnostics.
    Sorting by score means that applying an operating confidence threshold later
    is just a prefix selection.
    """
    keep, suppressed = nms_with_provenance(run["boxes"], run["scores"], iou_threshold)
    keep = np.array(keep, dtype=int)

    cross_tile_suppressed, same_tile_suppressed = 0, 0
    for s, k in suppressed:
        if run["tile_id"][s] != run["tile_id"][k]:
            cross_tile_suppressed += 1
        else:
            same_tile_suppressed += 1

    return {
        "boxes": run["boxes"][keep].reshape(-1, 4),
        "scores": run["scores"][keep].reshape(-1),
        "tile_id": run["tile_id"][keep].reshape(-1),
        "tile_boxes": run["tile_boxes"][keep].reshape(-1, 4),
        "n_pre_nms": int(len(run["scores"])),
        "n_suppressed_cross_tile": int(cross_tile_suppressed),
        "n_suppressed_same_tile": int(same_tile_suppressed),
    }


print("Offline NMS ready. Threshold used (frozen):", BEST_NMS_IOU)

In [ ]:
# =============================================================================
# CELL 22 - EVALUATION CORE: all_gt AND held_out
# =============================================================================
# all_gt   : classical evaluation, every GT box of the image counts.
#
# held_out : the GT instances that were shown to SAM3 as visual prompts are
#            REMOVED from the GT set, and predictions that fall on those prompt
#            plants are IGNORED (they are neither TP nor FP, they simply do not
#            exist for this evaluation). It answers: "after being shown a few
#            examples, how well does SAM3 find the REMAINING Rumex plants?"
#
# ORDER OF OPERATIONS (important, this is the agreed protocol):
#   1. match predictions to the evaluated (non-prompt) GT with the corrected
#      one-to-one matcher at EVAL_IOU_THRESHOLD = 0.50
#   2. every STILL UNMATCHED prediction whose best IoU with a PROMPT GT box is
#      >= PROMPT_IGNORE_IOU (0.50) becomes IGNORED
#   3. whatever is still unmatched is a false positive
#   Prompt GT boxes themselves are never counted as false negatives.
# =============================================================================

STATUS_FP, STATUS_TP, STATUS_IGNORED = 0, 1, 2


def split_gt_for_mode(gt_boxes, prompt_indices, mode):
    """
    Input : all GT boxes of the image, the indices used as visual prompts, mode
    Output: (evaluated_gt_boxes, prompt_gt_boxes)
            all_gt   -> (all boxes, empty)
            held_out -> (non-prompt boxes, prompt boxes)
    """
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    if mode == "all_gt":
        return gt_boxes, np.zeros((0, 4), dtype=np.float32)
    is_prompt = np.zeros(len(gt_boxes), dtype=bool)
    prompt_indices = np.asarray(prompt_indices, dtype=int)
    if len(prompt_indices):
        is_prompt[prompt_indices] = True
    return gt_boxes[~is_prompt], gt_boxes[is_prompt]


def evaluate_run_predictions(pred_boxes, pred_scores, eval_gt_boxes, prompt_gt_boxes,
                             eval_iou=EVAL_IOU_THRESHOLD, ignore_iou=PROMPT_IGNORE_IOU):
    """
    Evaluate ONE prediction set against ONE GT set.

    Output dict:
      status      (P,) int  - STATUS_TP / STATUS_FP / STATUS_IGNORED per prediction
      TP, FP, FN, n_ignored, n_eval_gt, n_pred
      precision, recall, F1
      IoU1 - mean IoU of the MATCHED prediction/GT pairs only
             ("when it finds a plant, how well is it localised?")
      IoU2 - sum of matched IoUs divided by the number of evaluated GT boxes
             ("localisation quality over ALL plants, missed ones count as 0")
      valid_for_macro - False when there is no GT left to evaluate (held_out runs
             in which every plant of the image was used as a prompt)
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)
    eval_gt_boxes = np.asarray(eval_gt_boxes, dtype=np.float32).reshape(-1, 4)
    prompt_gt_boxes = np.asarray(prompt_gt_boxes, dtype=np.float32).reshape(-1, 4)

    n_pred, n_eval_gt = len(pred_boxes), len(eval_gt_boxes)
    status = np.full(n_pred, STATUS_FP, dtype=np.int8) # start eveything as FP then valid matches TP then leftover predictions on prompt as ignored and everything still FP

    # step 1 - corrected one-to-one matching against the evaluated GT
    match = match_one_to_one(pred_boxes, pred_scores, eval_gt_boxes, eval_iou)
    status[match["pred_match_gt"] >= 0] = STATUS_TP # Mark matched predictions as TP

    # step 2 - ignore the leftovers that sit on a PROMPT plant
    # Therefore prompt-ignore logic applies only to predictions that failed to match evaluated GT
    if n_pred and len(prompt_gt_boxes):
        leftover = np.where(status == STATUS_FP)[0] # Find predictions that are still FP (unmatched preds)
        if len(leftover):
            best_prompt_iou = compute_iou_matrix(pred_boxes[leftover], prompt_gt_boxes).max(axis=1)
            status[leftover[best_prompt_iou >= ignore_iou]] = STATUS_IGNORED

    tp = int((status == STATUS_TP).sum())
    fp = int((status == STATUS_FP).sum())          # step 3 - the rest are FP
    n_ignored = int((status == STATUS_IGNORED).sum())
    fn = int(n_eval_gt - tp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / n_eval_gt if n_eval_gt > 0 else 0.0
    f1 = safe_f1(precision, recall)
    matched_ious = match["matched_ious"]
    iou1 = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2 = float(np.sum(matched_ious) / n_eval_gt) if n_eval_gt > 0 else 0.0

    return {
        "status": status, "pred_match_gt": match["pred_match_gt"],
        "TP": tp, "FP": fp, "FN": fn, "n_ignored": n_ignored,
        "n_eval_gt": n_eval_gt, "n_pred": n_pred,
        "precision": float(precision), "recall": float(recall), "F1": float(f1),
        "IoU1": iou1, "IoU2": iou2,
        "valid_for_macro": bool(n_eval_gt > 0),
    }


print("Evaluation core ready (modes:", EVALUATION_MODES, ")")

In [ ]:
# =============================================================================
# CELL 23 - AP50 AND AP50:95  (supervision.metrics.MeanAveragePrecision)
# =============================================================================
# WHY AP IS COMPUTED DIFFERENTLY FROM PRECISION / RECALL / F1
#
#   AP is an area under the precision-recall curve. That curve is produced by
#   walking through ALL detections ordered by confidence. Truncating the
#   detection list at an operating threshold would simply cut the tail off the
#   curve and report a smaller area - which says nothing about model quality.
#   Therefore AP always uses ALL saved predictions with score >= 0.30 (the SAM3
#   inference threshold), after the selected NMS.
#
#   Precision / recall / F1 / IoU1 / IoU2 describe ONE operating point: they
#   answer "if I deploy the detector with confidence >= c, what happens?".
#   Those use only the detections that survive the selected operating threshold.
#
#   Both numbers can appear in the same row - they answer different questions.
# =============================================================================

# just a helper function that converts NumPy boxes into the format expected by Supervision
def make_detections(boxes, scores=None):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    n = len(boxes)
    if scores is None:
        return sv.Detections(xyxy=boxes, class_id=np.zeros(n, dtype=int))
    return sv.Detections(xyxy=boxes,
                         confidence=np.asarray(scores, dtype=np.float32).reshape(-1),
                         class_id=np.zeros(n, dtype=int))


def compute_ap(pred_list, gt_list):
    """
    Input : two equally long lists of supervision Detections (predictions / GT).
            One entry = one evaluation episode (one run).
    Output: (AP50, AP50_95). NaN when there is no GT at all to evaluate.
    Passing several episodes at once gives the POOLED (dataset-level) AP, in which
    the detections of all episodes are ranked together in one PR curve.
    """
    if len(gt_list) == 0 or sum(len(g) for g in gt_list) == 0: # Check whether there is any GT
        return float("nan"), float("nan")
    try:
        # Create the Supervision metric (metric calculator, provides all pred and GTs, calculates the AP results)
        # Supervision exposes map50 for IoU=0.50 and map50_95 for IoUs 0.50:0.95
        result = MeanAveragePrecision().update(pred_list, gt_list).compute()
        ap50, ap5095 = float(result.map50), float(result.map50_95)
        # supervision returns -1.0 when a metric is undefined -> report NaN instead,
        # so it is excluded from means instead of dragging them down.
        return (ap50 if ap50 >= 0 else float("nan"),
                ap5095 if ap5095 >= 0 else float("nan"))
    except Exception as e:
        print("   (AP computation failed:", e, ")")
        return float("nan"), float("nan")


# Its job is not to calculate AP yet, Its job is to prepare: pred and GT for one run so they can later be given to compute_ap(...)
# nms_run: contains your detections after NMS
def ap_inputs_for_run(nms_run, eval_gt, prompt_gt):
    """
    Build the (prediction, GT) episode used for AP of ONE run.
    All post-NMS predictions with score >= SAM3_INFERENCE_THRESHOLD are used;
    in held_out mode the predictions that were IGNORED (they belong to prompt
    plants) are removed first, exactly like in the operating-point evaluation.
    """
    ev = evaluate_run_predictions(nms_run["boxes"], nms_run["scores"], eval_gt, prompt_gt) # marks each pred as TP,FP,IGNORED, it cares only about the ignored predictions for AP calculation bcz in held_out mode , predictions corresponding to prompt rumex must be removed before AP
    keep = ev["status"] != STATUS_IGNORED  # Remove ignored predictions
    # Convert remaining predictions to Supervision format
    return (make_detections(nms_run["boxes"][keep], nms_run["scores"][keep]),
            make_detections(eval_gt))


print("AP helpers ready (AP always uses every prediction >= "
      f"{SAM3_INFERENCE_THRESHOLD} after NMS).")

In [ ]:
# =============================================================================
# CELL 24 - OPERATING-POINT EVALUATION
# =============================================================================
# Choose one confidence threshold, remove predictions below it, then send the
# remaining predictions to CELL 22 for evaluation. Gives Precision, Recall,
# F1, IoU1, and IoU2 at one operating threshold.
# =============================================================================

def evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, confidence_threshold):
    """
    Input : nms_run  - output of apply_nms_to_run (sorted by score, high -> low) => contains the detections after NMS            eval_gt / prompt_gt - from split_gt_for_mode
            confidence_threshold - the operating point being tested
    Output: the dict of evaluate_run_predictions for the thresholded predictions.
    """
    keep = nms_run["scores"] >= confidence_threshold # boolean mask where True entries are selected and the False entries are excluded
    return evaluate_run_predictions(nms_run["boxes"][keep], nms_run["scores"][keep],
                                    eval_gt, prompt_gt)


print("Operating-point evaluation ready. Confidence threshold used (frozen):", BEST_CONFIDENCE)

In [ ]:
# =============================================================================
# CELL 25 - OPERATING CONFIGURATION (FROZEN, NO OFFLINE SWEEP)
# =============================================================================
# The offline confidence x NMS sweep (previous CELL 22) and the automatic
# "best configuration" selection built from that sweep (previous CELL 23) have
# been removed. BEST_CONFIDENCE and BEST_NMS_IOU are fixed in CELL 3 (both =
# 0.40) and are used directly, unchanged, by every cell from here on.
# =============================================================================

print("Operating configuration is frozen (no sweep performed):")
print(f"  Confidence threshold = {BEST_CONFIDENCE:.2f}")
print(f"  NMS IoU threshold    = {BEST_NMS_IOU:.2f}")

In [ ]:
# =============================================================================
# CELL 26 - RUN-LEVEL METRICS  (one run = one image x one anchor/prompt set)
# =============================================================================
# Everything is evaluated at the FROZEN configuration from CELL 25.
#   AP50 / AP50_95 : all post-NMS predictions >= 0.30, confidence-ranked
#   P / R / F1 / IoU1 / IoU2 / TP / FP / FN : only predictions >= BEST_CONFIDENCE
#
# Special case (held_out with no evaluable GT, i.e. every plant of the image was
# used as a prompt): the metrics are written as NaN and valid_for_macro = False so
# they are excluded from every mean/std, but TP/FN = 0 and the real FP count are
# kept, because such a run can still produce false positives that must show up in
# the pooled counts and in the confusion matrix.
# =============================================================================

run_rows = []
for run in runs:
    nms_run = apply_nms_to_run(run, BEST_NMS_IOU)
    for mode in EVALUATION_MODES:
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)

        # ---- AP: every prediction >= 0.30 after NMS (ignored ones removed) ----
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        ap50, ap5095 = compute_ap([p_det], [g_det])

        # ---- operating point -------------------------------------------------
        ev = evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, BEST_CONFIDENCE)
        valid = ev["valid_for_macro"]
        nan = float("nan")

        run_rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": run["image_ID"],
            "anchor_idx": run["anchor_idx"],
            "Prompt_ID": run["Prompt_ID"],
            "Prompt_Type": run["Prompt_Type"],
            "evaluation_mode": mode,
            "confidence_threshold": BEST_CONFIDENCE,
            "nms_iou_threshold": BEST_NMS_IOU,
            "n_gt_total": int(len(run["gt_boxes"])),
            "n_prompt_gt": int(len(run["prompt_indices"])) if mode == "held_out" else 0,
            "n_eval_gt": ev["n_eval_gt"],
            "n_predictions": ev["n_pred"],
            "n_ignored_predictions": ev["n_ignored"],
            "AP50": ap50 if valid else nan,
            "AP50_95": ap5095 if valid else nan,
            "precision": ev["precision"] if valid else nan,
            "recall": ev["recall"] if valid else nan,
            "F1": ev["F1"] if valid else nan,
            "IoU1": ev["IoU1"] if valid else nan,
            "IoU2": ev["IoU2"] if valid else nan,
            "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
            "valid_for_macro": valid,
        })

run_level_df = pd.DataFrame(run_rows)
RUN_LEVEL_CSV = os.path.join(METRICS_DIR, "run_level_metrics.csv")
run_level_df.to_csv(RUN_LEVEL_CSV, index=False)

print(f"Run-level metrics: {len(run_level_df)} rows -> {RUN_LEVEL_CSV}")
for mode in EVALUATION_MODES:
    sub = run_level_df[run_level_df["evaluation_mode"] == mode]
    print(f"  {mode:9s}: {len(sub)} runs, "
          f"{int(sub['valid_for_macro'].sum())} valid for macro averaging, "
          f"F1_mean={sub['F1'].mean():.4f}")

In [ ]:
# =============================================================================
# CELL 27 - IMAGE-LEVEL METRICS
# =============================================================================
# All anchor runs of the same image are averaged into ONE value per image and per
# evaluation mode. The std here is the spread BETWEEN the different anchor/prompt
# selections of the SAME image, i.e. "how sensitive is the result to which plant
# was used as the visual prompt?".
# NaN rows (held_out runs with no evaluable GT) are ignored by pandas mean/std.
# std is NaN when an image has only one valid run - that is expected.
# =============================================================================

METRIC_COLUMNS = ["AP50", "AP50_95", "precision", "recall", "F1", "IoU1", "IoU2"]

image_rows = []
for (image_id, mode), grp in run_level_df.groupby(["image_ID", "evaluation_mode"]):
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "image_ID": image_id,
        "evaluation_mode": mode,
        "confidence_threshold": BEST_CONFIDENCE,
        "nms_iou_threshold": BEST_NMS_IOU,
        "n_runs_total": int(len(grp)),
        "n_runs_valid_for_macro": int(grp["valid_for_macro"].sum()),
        "TP_sum": int(grp["TP"].sum()),
        "FP_sum": int(grp["FP"].sum()),
        "FN_sum": int(grp["FN"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = grp[col].mean()      # NaNs skipped automatically
        row[f"{col}_std"] = grp[col].std()        # sample std (ddof=1)
    image_rows.append(row)

image_level_df = pd.DataFrame(image_rows).sort_values(
    ["evaluation_mode", "image_ID"]).reset_index(drop=True)
IMAGE_LEVEL_CSV = os.path.join(METRICS_DIR, "image_level_metrics.csv")
image_level_df.to_csv(IMAGE_LEVEL_CSV, index=False)

print(f"Image-level metrics: {len(image_level_df)} rows -> {IMAGE_LEVEL_CSV}")
print(image_level_df.groupby("evaluation_mode")[
    ["AP50_mean", "precision_mean", "recall_mean", "F1_mean", "IoU1_mean", "IoU2_mean"]
].mean().to_string())

In [ ]:
# =============================================================================
# CELL 28 - EXPERIMENT-LEVEL SUMMARY
# =============================================================================
# Computed from the IMAGE-LEVEL values, not from the raw run rows, so that every
# UAV image contributes exactly the same weight regardless of how many GT boxes
# (and therefore how many anchor runs) it contains.
# The std here is the variation BETWEEN UAV images.
# =============================================================================

summary_rows = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_exemplars": N_EXEMPLARS,
        "use_tiling": USE_TILING,
        "tile_size": TILE_SIZE,
        "overlap": OVERLAP,
        "add_global_context_pass": ADD_GLOBAL_CONTEXT_PASS,
        "confidence_threshold": BEST_CONFIDENCE,
        "nms_iou_threshold": BEST_NMS_IOU,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
        "n_images": int(sub["image_ID"].nunique()),
        "n_runs": int(sub["n_runs_total"].sum()),
        "n_runs_valid_for_macro": int(sub["n_runs_valid_for_macro"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = sub[f"{col}_mean"].mean()
        row[f"{col}_std"] = sub[f"{col}_mean"].std()   # spread between images
    summary_rows.append(row)

experiment_summary_df = pd.DataFrame(summary_rows)
EXPERIMENT_SUMMARY_CSV = os.path.join(METRICS_DIR, "experiment_summary.csv")
experiment_summary_df.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)

print(f"Experiment summary -> {EXPERIMENT_SUMMARY_CSV}\n")
print(experiment_summary_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 29 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# CELL 26 computes AP separately for each run.
# This cell gives all runs to the AP evaluator together and computes one
# overall AP for the experiment (one result row per evaluation mode).
# =============================================================================
# =============================================================================
# CELL 29 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# This is NOT the mean of the image-level AP values. All runs are handed to
# supervision as evaluation EPISODES at once, so every detection of the whole
# dataset is ranked in ONE precision-recall curve.
#
# NOTE for the thesis text: because every GT box of an image becomes an anchor
# once, the same UAV image appears in several episodes (once per prompt set).
# The pooled AP is therefore computed over "pooled evaluation episodes", not over
# unique images - it measures the ranking quality of the whole experiment.
# =============================================================================

# At IoU 0.50, each detection is judged as TP or FP against the GT of its own episode
# Then, conceptually, the confidence-ranked detection sequence across the experiment is: ... As detections
# are accumulated in confidence order, overall precision and recall change, producing the experiment-level PR curve

dataset_rows = []
for mode in EVALUATION_MODES:
    pred_list, gt_list, images_used = [], [], set()
    for run in runs:
        nms_run = apply_nms_to_run(run, BEST_NMS_IOU)
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt) # Prepare this run's AP episode
        pred_list.append(p_det)
        gt_list.append(g_det)
        images_used.add(run["image_ID"])
    ap50, ap5095 = compute_ap(pred_list, gt_list) # After ALL runs, calculate AP (after the for run in runs loop)
    dataset_rows.append({
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "n_images": len(images_used),
        "n_runs": len(runs),
        "confidence_used_for_AP": SAM3_INFERENCE_THRESHOLD,   # AP always uses >= 0.30
        "nms_iou_threshold": BEST_NMS_IOU,
        "dataset_AP50": ap50,
        "dataset_AP50_95": ap5095,
    })
    del pred_list, gt_list
    gc.collect()

dataset_ap_df = pd.DataFrame(dataset_rows)
DATASET_AP_CSV = os.path.join(METRICS_DIR, "dataset_ap_metrics.csv")
dataset_ap_df.to_csv(DATASET_AP_CSV, index=False)

print(f"Dataset pooled AP -> {DATASET_AP_CSV}\n")
print(dataset_ap_df.to_string(index=False))
print("\nFor comparison, the MEAN of the image-level AP50 values "
      "(a different quantity):")
print(image_level_df.groupby("evaluation_mode")["AP50_mean"].mean().to_string())

In [ ]:
# =============================================================================
# CELL 30 - DATASET-LEVEL CONFUSION MATRICES
# =============================================================================
# One class (Rumex) plus a background row/column:
#     Actual Rumex      -> Predicted Rumex      = TP
#     Actual Rumex      -> Predicted Background = FN  (missed plants)
#     Actual Background -> Predicted Rumex      = FP  (spurious detections)
#     Actual Background -> Predicted Background = not defined for detection
#                                                 (there are no true negatives)
# Counts are pooled over every run at the frozen configuration. In held_out mode
# the prompt plants and the detections that were ignored do not appear anywhere.
# =============================================================================

def plot_confusion_matrix(tp, fp, fn, title, png_path):
    """2x2 detection confusion matrix; the background/background cell stays empty."""
    matrix = np.array([[tp, fn], [fp, np.nan]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="Blues")
    ax.set_xticks([0, 1], ["Predicted\nRumex", "Predicted\nBackground"])
    ax.set_yticks([0, 1], ["Actual\nRumex", "Actual\nBackground"])
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", "n/a\n(no true\nnegatives)"]]
    vmax = np.nanmax(matrix) if np.nanmax(matrix) > 0 else 1.0
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            colour = "white" if (not np.isnan(value) and value > 0.5 * vmax) else "black"
            ax.text(j, i, labels[i][j], ha="center", va="center",
                    color=colour, fontsize=11)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.show()
    plt.close(fig)


confusion_summary = []
for mode in EVALUATION_MODES:
    sub = run_level_df[run_level_df["evaluation_mode"] == mode]
    tp, fp, fn = int(sub["TP"].sum()), int(sub["FP"].sum()), int(sub["FN"].sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = safe_f1(precision, recall)

    cm_df = pd.DataFrame(
        [[tp, fn], [fp, np.nan]],
        index=["actual_rumex", "actual_background"],
        columns=["predicted_rumex", "predicted_background"],
    )
    csv_path = os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.csv")
    cm_df.to_csv(csv_path)

    png_path = os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.png")
    plot_confusion_matrix(
        tp, fp, fn,
        f"{EXPERIMENT_NAME} - {mode}\nconf={BEST_CONFIDENCE:.2f}, "
        f"NMS IoU={BEST_NMS_IOU:.2f}, eval IoU={EVAL_IOU_THRESHOLD:.2f}",
        png_path)

    confusion_summary.append({
        "experiment_name": EXPERIMENT_NAME, "evaluation_mode": mode,
        "TP": tp, "FP": fp, "FN": fn,
        "precision_micro": precision, "recall_micro": recall, "F1_micro": f1,
        "confidence_threshold": BEST_CONFIDENCE, "nms_iou_threshold": BEST_NMS_IOU,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
    })
    print(f"{mode:9s}: TP={tp}  FP={fp}  FN={fn}  "
          f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

confusion_summary_df = pd.DataFrame(confusion_summary)
confusion_summary_df.to_csv(
    os.path.join(CONFUSION_MATRIX_DIR, "confusion_matrix_summary.csv"), index=False)
print("\nConfusion matrices saved to:", CONFUSION_MATRIX_DIR)

In [ ]:
# =============================================================================
# CELL 31 - FINAL OUTPUT SUMMARY
# =============================================================================

print("=" * 78)
print(f"EXPERIMENT {EXPERIMENT_NAME} - FINAL SUMMARY")
print("=" * 78)
print(f"Prompts per run          : {N_EXEMPLARS} ({PROMPT_TYPE})")
print(f"Tiling                   : {USE_TILING}  (tile={TILE_SIZE}px, overlap={OVERLAP}px)")
print(f"SAM3 inference threshold : {SAM3_INFERENCE_THRESHOLD} (executed once per image x anchor x tile)")
print(f"Global context pass      : {ADD_GLOBAL_CONTEXT_PASS} (downscale={GLOBAL_DOWNSCALE})")
print(f"Selected operating point : confidence={BEST_CONFIDENCE:.2f}, NMS IoU={BEST_NMS_IOU:.2f}")
print(f"Evaluation IoU           : {EVAL_IOU_THRESHOLD:.2f}")
print(f"Runs / images            : {len(runs)} runs over "
      f"{run_level_df['image_ID'].nunique()} images")
print("-" * 78)
print("EXPERIMENT-LEVEL RESULTS (mean over images, std between images)")
show = ["evaluation_mode", "AP50_mean", "AP50_std", "AP50_95_mean", "precision_mean",
        "recall_mean", "F1_mean", "F1_std", "IoU1_mean", "IoU2_mean"]
print(experiment_summary_df[show].to_string(index=False))
print("-" * 78)
print("POOLED DATASET AP")
print(dataset_ap_df[["evaluation_mode", "dataset_AP50", "dataset_AP50_95"]].to_string(index=False))
print("-" * 78)
print("POOLED CONFUSION COUNTS")
print(confusion_summary_df[["evaluation_mode", "TP", "FP", "FN",
                            "precision_micro", "recall_micro", "F1_micro"]].to_string(index=False))
print("=" * 78)

print("\nFiles written under", RESULTS_ROOT)
for root, dirs, files in os.walk(RESULTS_ROOT):
    depth = root.replace(RESULTS_ROOT, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    if os.path.basename(root) == "raw_detections":
        npz_files = [f for f in files if f.endswith(".npz")]
        for f in sorted(files):
            if not f.endswith(".npz"):
                print("  " * (depth + 1) + f)
        print("  " * (depth + 1) + f"[{len(npz_files)} run NPZ files]")
    else:
        for f in sorted(files):
            print("  " * (depth + 1) + f)